# SIFLOW · run_1 · MDLM data prep

Streams + tokenizes OpenWebText into length-256 GPT-2 chunks (train + a disjoint val split used as the MAUVE reference). The MDLM teacher is cheap enough to run live, so **no simplex cache is needed** for the primary study.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_0 passed

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
from transformers import AutoTokenizer
from siflow.data import build_token_chunks

tok = AutoTokenizer.from_pretrained("gpt2")
N_TRAIN = 200_000   # ~51M tokens; reused across epochs
N_VAL   = 5_000

n = build_token_chunks(tok, 256, N_TRAIN, f"{BASE}/data/owt_gpt2_256.npy",
                       dataset="Skylion007/openwebtext", split="train")
print("train chunks:", n)
nv = build_token_chunks(tok, 256, N_VAL, f"{BASE}/data/owt_gpt2_val.npy",
                        dataset="Skylion007/openwebtext", split="train", skip_seqs=N_TRAIN)
print("val chunks:", nv)

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---

print('saved to', BASE)

Data lives on Drive at `{BASE}/data/`. Proceed to **run_2** to train the SIFLOW head.